In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re

def read_files_from_folder(txt_folder):
    texts = []
    for filename in os.listdir(txt_folder):
        if filename.endswith('.txt'):
            file_path = os.path.join(txt_folder, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                text = file.read()
                texts.append((filename, text))
    return texts

txt_folder_path = "/content/drive/MyDrive/TA/0. Crawling/txt_putusan/kasasiMA"
texts = read_files_from_folder(txt_folder_path)

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,;:() \n]', '', text)
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)
    return text.strip()

In [ ]:
import pandas as pd

def extract_features(filename, text):
    features = {
        "filename": filename,
        "tanggal_putusan": None,
        "nama_hakim": None,
        "nomor_perkara": None,
        "ringkasan_argumen": None,
        "putusan_pertama": None,
        "putusan_banding": None,
        "hasil_kasasi": None
    }

    # Nomor Perkara
    nomor_perkara_pattern = r'\b\s*(?:Hal(?:aman|\.?|\.?l|m|\.?l\.?|m\.?)?)\s+\s*(?:Penetapan|Put(?:usan|\.?)?)?\s+\s*(?:No(?:mor|\.?)?)\s+\s*([0-9A-Za-z./\s])\b'
    match = re.search(nomor_perkara_pattern, text, re.IGNORECASE)
    if match:
        features["nomor_perkara"] = re.sub(r'\s+', ' ', match.group(1)).strip()

    # Putusan Pertama
    putusan_pertama_pattern = r'\bPengadilan\s*Negeri\s*([A-Za-z\s]+)\s*telah\s*memberikan\s*Putusan\s*(?:No(?:mor|\.?)?)\s+\s*([0-9A-Za-z./\s]+PN\s+[A-Za-z]+)'
    match = re.search(putusan_pertama_pattern, text, re.IGNORECASE)
    if match:
        pengadilan_negeri = re.sub(r'\s+', ' ', match.group(1)).strip()
        putusan_nomor = re.sub(r'\s+', ' ', match.group(2)).strip()
        features["putusan_pertama"] = f"{pengadilan_negeri} - {putusan_nomor}"

    # Putusan Banding
    putusan_banding_pattern = r'\bPengadilan\s*Tinggi\s*([A-Za-z\s]+)\s*dengan\s*Putusan\s*(?:No(?:mor|\.?)?)\s*([0-9A-Za-z./\s]+PT\s+[A-Za-z]+)'
    match = re.search(putusan_banding_pattern, text, re.IGNORECASE)
    if match:
        pengadilan_tinggi = re.sub(r'\s+', ' ', match.group(1)).strip()
        putusan_nomor = re.sub(r'\s+', ' ', match.group(2)).strip()
        features["putusan_banding"] = f"{pengadilan_tinggi} - {putusan_nomor}"

    cleaned_text = clean_text(text)

    # Tanggal Putusan
    tgl_pattern = r'\bDemikian(lah)?\s*diputuskan\s*dalam\s*(rapat\s*)?(musyawarah|permusyawaratan|sidang)\s*Majelis\s*Hakim(\s*Pengadilan\s*Negeri\s*\w+\s*\w+)?\s*(.*?)\s*pada\s*hari\s*(\w+),?\s*tanggal\s*(\d{1,2}\s*\w+\s*\d{4})\b'
    match = re.search(tgl_pattern, cleaned_text, re.IGNORECASE)
    if match:
        features["tanggal_putusan"] = re.sub(r'\s+', ' ', match.group(7)).strip()

    # Nama Hakim
    hakim_pattern = r'\bDemikian(?:lah)?\s*diputuskan\s*dalam\s*(?:rapat\s*)?(?:musyawarah|permusyawaratan|sidang)\s*Majelis\s*Hakim(?:\s*Pengadilan\s*Negeri\s*\w+\s*\w+)?\s*.*?\s*pada\s*hari\s*\w+,\s*tanggal\s*\d{1,2}\s*\w+\s*\d{4}\s*oleh(?:[:,\s]*)\s*([\w\s\.,]+?)(?=,\s*(?:Hakim|yang ditetapkan oleh|\d{4}))'
    match = re.search(hakim_pattern, cleaned_text, re.IGNORECASE)
    if match:
        features["nama_hakim"] = re.sub(r',? oleh\b.*', '', match.group(1)).strip()

    # Ringkasan Argumen
    argumen_pattern = r'\bMenimbang,\s*bahwa(?:[^.]*?Penggugat(?:[^.]*?memohon|[^.]*?kepada\s*Pengadilan(?:\s*Negeri)?)[^.]*?\.)'
    match = re.search(argumen_pattern, cleaned_text, re.IGNORECASE | re.DOTALL)
    if match:
        features["ringkasan_argumen"] = re.sub(r'\s+', ' ', match.group(0)).strip()

    # Hasil Kasasi
    hasil_kasasi_pattern = r'\bMenolak\s*permohonan\s*kasasi|Menerima\s*permohonan\s*kasasi'
    match = re.search(hasil_kasasi_pattern, cleaned_text, re.IGNORECASE)
    if match:
        features["hasil_kasasi"] = re.sub(r'\s+', ' ', match.group(0)).strip()

    return features

In [ ]:
all_features = [extract_features(filename, text) for filename, text in texts]

In [ ]:
df = pd.DataFrame(all_features)
df

,filename,tanggal_putusan,nama_hakim,nomor_perkara,ringkasan_argumen,putusan_pertama,putusan_banding,hasil_kasasi
0,putusan_4271_k_pdt_2022_20241025042336.pdf_202...,None,"Dr. H. Hamdi, S.H., M.Hum.",None,"Menimbang, bahwaberdasarkansuratsuratyang bers...",None,Kalimantan Timur diSamarinda - 129/PDT/2019/PT...,Menolak permohonan kasasi
1,putusan_4662_k_pdt_2022_20241025042341.pdf_202...,19 Desember 2022,"Dr. H. Panji Widagdo, S.H.,M.H.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Cilacap - 56/Pdt.G/2021/PN Clp,None,Menerima permohonan kasasi
2,putusan_4095_k_pdt_2022_20241025042410.pdf_202...,14 Desember 2022,"Dr. Ibrahim, S.H., M.H., LL.M.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",None,Manado - 52/PDT/2022/PT MND,menolak permohonan kasasi
3,putusan_4668_k_pdt_2022_20241025042429.pdf_202...,19 Desember 2022,"Dr. H. Panji Widagdo, S.H., M.H.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Kuala Kapuas - 14/Pdt.G/2019/PN Klk,None,Menerima permohonan kasasi
4,putusan_4451_k_pdt_2022_20241025042440.pdf_202...,15 Desember 2022,"Dr. H. Zahrul Rabain, S.H., M.H.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Kotabumi - 13/Pdt.G/2020/PN Kbu,Tanjungkarang - 31/Pdt/2021/PT TJK,Menerima permohonan kasasi
...,...,...,...,...,...,...,...,...
2408,putusan_4392_k_pdt_2023_20241025001407.pdf_202...,21 Desember 2023,"Dr. Yakup Ginting, S.H., C.N.,M.Kn.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Kayu Agung - 25/Pdt.G/2022/PN Kag,Palembang - 49/PDT/2023/PT PLG,Menolak permohonan kasasi
2409,putusan_3989_k_pdt_2023_20241025001425.pdf_202...,21 Desember 2023,"Dr. Yakup Ginting, S.H., C.N.,M.Kn.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Barru - 7/Pdt.G/2022/PN Bar,Makassar - 75/PDT/2023/PT MKS,Menerima permohonan kasasi
2410,putusan_4651_k_pdt_2023_20241025001427.pdf_202...,21 Desember 2023,"Dr. H. Panji Widagdo, S.H., M.H.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Jakarta Pusat - 478/Pdt.G/2020/PN Jkt,None,Menerima permohonan kasasi
2411,putusan_4256_k_pdt_2023_20241025001443.pdf_202...,21 Desmeber 2023,"Dr. Yakup Ginting, S.H., C.N.,M.Kn.",None,"Menimbang, bahwa berdasarkan suratsurat yang b...",Pangkalan Bun - 37/Pdt.G/2022/PN Pbu,None,menolakpermohonan kasasi


In [ ]:
import csv

output_csv = '/content/drive/MyDrive/TA/0. Crawling/extracted_features.csv'
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ["filename", "tanggal_putusan", "nama_hakim", "nomor_perkara", "ringkasan_argumen", "putusan_pertama", "putusan_banding", "hasil_kasasi"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for features in all_features:
        writer.writerow(features)